In [66]:
from random import seed,randint
from numpy import array
from math import ceil,log10,sqrt,log
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,TimeDistributed,RepeatVector

In [67]:
def random_sum_pairs(n_exmples,n_numbers,largest):
    x,y=[],[]
    for i in range(n_exmples):
        in_pattern=[randint(1,largest) for _ in range(n_numbers)]
        out_pattern=sum(in_pattern)
        x.append(in_pattern)
        y.append(out_pattern)
    return x,y

In [68]:
x,y=random_sum_pairs(100,4,20)
print(x)

[[7, 2, 4, 17], [18, 4, 16, 7], [3, 16, 1, 20], [11, 6, 9, 8], [1, 16, 10, 9], [12, 15, 1, 5], [1, 15, 3, 6], [15, 18, 15, 11], [5, 3, 18, 10], [19, 10, 2, 9], [4, 16, 12, 15], [6, 19, 4, 11], [12, 15, 5, 19], [9, 4, 20, 3], [11, 8, 18, 20], [6, 13, 19, 15], [13, 2, 9, 10], [11, 1, 5, 3], [3, 1, 19, 19], [11, 7, 8, 7], [20, 17, 10, 7], [16, 8, 18, 9], [8, 7, 10, 11], [11, 15, 11, 14], [7, 10, 13, 5], [18, 15, 14, 16], [1, 10, 18, 5], [7, 17, 7, 2], [17, 8, 14, 19], [1, 8, 4, 12], [6, 10, 11, 15], [17, 4, 15, 20], [17, 16, 10, 3], [13, 11, 1, 13], [9, 2, 19, 16], [20, 2, 11, 5], [18, 17, 17, 3], [4, 16, 6, 1], [7, 16, 18, 9], [7, 9, 19, 10], [8, 15, 13, 3], [11, 7, 13, 7], [17, 14, 16, 14], [10, 19, 13, 9], [11, 6, 9, 13], [18, 14, 17, 12], [10, 2, 10, 12], [12, 9, 14, 8], [15, 15, 9, 16], [18, 2, 14, 9], [7, 16, 3, 6], [5, 6, 5, 14], [10, 8, 14, 10], [7, 11, 14, 16], [20, 16, 8, 7], [6, 3, 12, 12], [16, 10, 2, 8], [1, 10, 7, 6], [17, 12, 11, 12], [8, 13, 14, 1], [9, 20, 15, 3], [13, 11

In [69]:
print(y)

[30, 45, 40, 34, 36, 33, 25, 59, 36, 40, 47, 40, 51, 36, 57, 53, 34, 20, 42, 33, 54, 51, 36, 51, 35, 63, 34, 33, 58, 25, 42, 56, 46, 38, 46, 38, 55, 27, 50, 45, 39, 38, 61, 51, 39, 61, 34, 43, 55, 43, 32, 30, 42, 48, 51, 33, 36, 24, 52, 36, 47, 52, 48, 48, 47, 42, 29, 49, 52, 41, 33, 48, 56, 36, 37, 26, 34, 33, 31, 58, 36, 70, 44, 42, 23, 71, 43, 42, 38, 47, 42, 45, 31, 36, 44, 37, 38, 54, 35, 30]


In [70]:
ceil(log10(20+1))

2

In [71]:
def pairs_to_string(x,y,n_numbers,largest):
    max_length=n_numbers*ceil(log10(largest+1))+n_numbers-1
    xstr=[]
    for p in x:
        strp='+'.join([str(n) for n in p])
        strp=''.join([' ' for _ in range(max_length-len(strp))])+strp
        xstr.append(strp)
    max_length=ceil(log10(n_numbers*(largest+1)))
    ystr=[]
    for p in y:
        strp=str(p)
        strp=''.join([' ' for _ in range(max_length-len(strp))])+strp
        ystr.append(strp)
    return xstr,ystr


In [72]:
xstr,ystr=pairs_to_string(x,y,4,20)

In [73]:
def integer_encode(x,y,alphabet):
    char_to_int=dict((c,i) for i,c in enumerate(alphabet))
    print(char_to_int)
    Xenc=[]
    for p in x:
        integer_encoded=[char_to_int[char]for char in p]
        Xenc.append(integer_encoded)
    yenc=[]
    for p in y:
        integer_encoded=[char_to_int[char]for char in p]
        yenc.append(integer_encoded)
    return Xenc,yenc

In [74]:
integer_encode(xstr,ystr,['1','2','3','4','5','6','7','8','9','0',' ','+'])

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}


([[10, 10, 10, 6, 11, 1, 11, 3, 11, 0, 6],
  [10, 10, 0, 7, 11, 3, 11, 0, 5, 11, 6],
  [10, 10, 2, 11, 0, 5, 11, 0, 11, 1, 9],
  [10, 10, 10, 0, 0, 11, 5, 11, 8, 11, 7],
  [10, 10, 0, 11, 0, 5, 11, 0, 9, 11, 8],
  [10, 10, 0, 1, 11, 0, 4, 11, 0, 11, 4],
  [10, 10, 10, 0, 11, 0, 4, 11, 2, 11, 5],
  [0, 4, 11, 0, 7, 11, 0, 4, 11, 0, 0],
  [10, 10, 4, 11, 2, 11, 0, 7, 11, 0, 9],
  [10, 10, 0, 8, 11, 0, 9, 11, 1, 11, 8],
  [10, 3, 11, 0, 5, 11, 0, 1, 11, 0, 4],
  [10, 10, 5, 11, 0, 8, 11, 3, 11, 0, 0],
  [10, 0, 1, 11, 0, 4, 11, 4, 11, 0, 8],
  [10, 10, 10, 8, 11, 3, 11, 1, 9, 11, 2],
  [10, 0, 0, 11, 7, 11, 0, 7, 11, 1, 9],
  [10, 5, 11, 0, 2, 11, 0, 8, 11, 0, 4],
  [10, 10, 0, 2, 11, 1, 11, 8, 11, 0, 9],
  [10, 10, 10, 0, 0, 11, 0, 11, 4, 11, 2],
  [10, 10, 2, 11, 0, 11, 0, 8, 11, 0, 8],
  [10, 10, 10, 0, 0, 11, 6, 11, 7, 11, 6],
  [10, 1, 9, 11, 0, 6, 11, 0, 9, 11, 6],
  [10, 10, 0, 5, 11, 7, 11, 0, 7, 11, 8],
  [10, 10, 7, 11, 6, 11, 0, 9, 11, 0, 0],
  [0, 0, 11, 0, 4, 11, 0, 0, 11, 0,

In [76]:
def one_hot_encode(x,y,max_int):
    xenc=[]
    for p in x:
        pattern=[]
        for index in p:
            vector=[0 for _ in range(max_int)]
            vector[index]=1
            pattern.append(vector)
        xenc.append(pattern)
    yenc=[]
    for p in y:
        pattern=[]
        for index in p:
            vector=[0 for _ in range(max_int)]
            vector[index]=1
            pattern.append(vector)
        yenc.append(pattern)
    return xenc,yenc

In [77]:
import numpy as np
def generate_data(n_samples,n_numbers,largest,alphabet):
    x,y=random_sum_pairs(n_samples,n_numbers,largest)
    x,y=pairs_to_string(x,y,n_numbers,largest)
    x,y=integer_encode(x,y,alphabet)
    x,y=one_hot_encode(x,y,len(alphabet))
    x,y=np.array(x),np.array(y)
    return x,y

In [84]:
from numpy import argmax
import numpy as np
def integer_decode(seq,alphabet):
    int_to_char=dict((i,c)for i,c in enumerate(alphabet))
    strings=[]
    for p in seq:
        string=int_to_char[argmax(p)]
        strings.append(string)
    return ''.join(strings)

In [85]:
seed(1)
n_sample=1000
n_numbers=3
largest=20
alphabets=['1','2','3','4','5','6','7','8','9','0',' ','+']
n_chars=len(alphabets)
n_in_seq_length=n_numbers*ceil(log10(largest+1))+n_numbers-1
n_out_seq_length=ceil(log10(largest+1))


In [86]:
n_out_seq_length

2

In [88]:
n_in_seq_length

8

In [89]:
x,y=generate_data(n_sample,n_numbers,largest,alphabets)

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}


In [90]:
x.shape

(1000, 8, 12)

In [91]:
y.shape

(1000, 2, 12)

In [92]:
x[0]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [98]:
model=Sequential()
model.add(LSTM(128,input_shape=(n_in_seq_length,n_chars)))
model.add(RepeatVector(n_out_seq_length))
model.add(LSTM(64,return_sequences=True))
model.add(TimeDistributed(Dense(n_chars,activation='softmax')))

In [99]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])


In [100]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 128)            │        72,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 2, 64)          │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 2, 12)          │           780 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,380 (478.05 KB)

 Trainable params: 122,380 (478.05 KB)

 Non-trainable params: 0 (0.00 B)

In [101]:
for i in range(200):
    x,y=generate_data(n_sample,n_numbers,largest,alphabets)
    print('epoch: ',i)
    model.fit(x,y,epochs=1,batch_size=50)

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  0
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2000 - loss: 2.4058
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  1
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2345 - loss: 2.1577
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  2
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2295 - loss: 1.9575
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  3
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2955 - loss: 1.8373
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, '+': 11}
epoch:  4
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3255 - loss: 1.7529
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '0': 9, ' ': 10, 